# 07. 대본 LLM 의역 + 검증 (파일럿 20클립) — **Colab GPU**

템플릿 대본에서 **판정 발화(call)와 해설위원 코멘트(comment, distract)만** 의역한다. **Hard 트랙만** 의역하고 Easy는 템플릿 그대로.
이닝·투수·타자 소개, 카운트 재언급, 공수 교대는 정답 단서라 그대로 둔다. 규칙과 프롬프트는 `kbolisten/paraphrase.py`.

| 단계 | 모델 | 내용 |
|---|---|---|
| 1. 생성 | Qwen3-32B | 클립 하나를 한 번에 의역 → 형식 검사 → 탈락한 줄만 최대 2번 다시 생성 |
| 2. 검증 | Gemma 3 27B | 원문·정답을 모르는 채 문장만 보고 판정 라벨 분류 → 원래 판정과 같아야 통과 |

생성과 검증을 다른 계열 모델로 하는 이유: 같은 모델이면 같은 착각을 같이 해서 오류가 통과한다.
끝까지 실패한 줄은 템플릿 원문을 쓴다 (클립은 버리지 않음).

**실행 전 준비**
- 새 런타임 권장 (vLLM이 PyTorch 버전을 바꿔서 qwen-tts와 꼬일 수 있음)
- `kbolisten/`은 Drive for desktop으로 동기화한 폴더(`Othercomputers/*/kbolisten`)에서 자동으로 불러온다
- Gemma는 승인이 필요한 모델: huggingface.co/google/gemma-3-27b-it 에서 라이선스 동의 →
  Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN` 등록
- 두 모델이 GPU에 같이 안 올라가서 **1단계 뒤 커널 재시작** → 셀 2 → 2단계 순서로 실행

**결과물 (Drive `kbo-listen/paraphrase_pilot/`)**: `generated.jsonl`(1단계), `scripts.jsonl`(최종 대본), `report.json`

In [ ]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6

In [1]:
# 공통 준비: 1단계·2단계 모두 커널 시작 후 이 셀부터 실행 (샘플링·대본은 시드 고정이라 매번 같음)
import glob
import json
import os
import random
import sys
import zlib
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

from google.colab import drive

# kbolisten은 Drive for desktop으로 동기화한 로컬 폴더에서 불러온다 (매번 업로드할 필요 없음)
drive.mount("/content/drive")
_found = glob.glob("/content/drive/Othercomputers/*/kbolisten")
assert _found, "Drive '컴퓨터' 영역에서 kbolisten을 못 찾음 → Drive for desktop 동기화 확인"
sys.path.insert(0, os.path.dirname(_found[0]))

from kbolisten import paraphrase as P
from kbolisten.script import KNOBS, render_clip

GEN_MODEL = "Qwen/Qwen3-32B"            # 생성: 라이선스(Apache 2.0) 문제 없는 모델 중 한국어가 가장 자연스러움
JUDGE_MODEL = "google/gemma-3-27b-it"   # 검증: 생성과 다른 계열


@contextmanager
def real_stdout():
    """vLLM은 엔진을 띄울 때 sys.stdout.fileno()를 부르는데, 노트북 커널의 출력 스트림은 fileno가 없어서 실패한다.
    엔진을 만드는 동안만 원래 터미널 출력으로 바꿔 둔다 (이 동안의 로그는 셀이 아니라 런타임 로그로 감)"""
    saved = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        yield
    finally:
        sys.stdout, sys.stderr = saved


DRIVE = Path("/content/drive/MyDrive/kbo-listen")
OUT = DRIVE / "paraphrase_pilot"
OUT.mkdir(parents=True, exist_ok=True)

VOICES = json.loads((DRIVE / "voices" / "voices.json").read_text(encoding="utf-8"))
CASTERS = sorted(v for v, info in VOICES.items() if info["role"] == "caster")
ANALYSTS = sorted(v for v, info in VOICES.items() if info["role"] == "analyst")
with (DRIVE / "labels_all.jsonl").open(encoding="utf-8") as f:
    clips = [json.loads(line) for line in f]

# 06과 같은 샘플링 (같은 시드 → 같은 20클립). 본 데이터셋 노트북에서는 공용 모듈로 옮길 예정
SPECIAL = {"pitch_clock", "hbp", "ibb", "walk_award", "runner_out"}
RARE = {"pitch_clock", "walk_award", "runner_out"}
MAX_PITCHES = 35
PILOT = {"train": (3, 3), "dev": (2, 2), "test": (5, 5)}
# 의역은 Hard만: 두 트랙에 다 했더니(v4) Easy도 키워드로 안 풀려서 트랙 차이가 사라졌음
# Easy = 템플릿 그대로(키워드로 풀 수 있는 기본), Hard = 의역 + 판정 생략 + 방해 코멘트
TRACK_KNOBS = {
    "easy": {"p_omit": 0.0, "p_back": 0.10, "p_comment": 0.15, "p_comment_pa": 0.6, "p_distract": 0.0,
             "paraphrase": False},
    "hard": {"p_omit": 0.5, "p_back": 0.15, "p_comment": 0.35, "p_comment_pa": 0.7, "p_distract": 0.5,
             "paraphrase": True},
}


def sample(clips, plan, seed=42):
    rng = random.Random(seed)
    ok = [c for c in clips if c["n_pitches"] <= MAX_PITCHES]
    picked, used = [], set()

    def take(pool, n, track):
        pool = [c for c in pool if c["clip_id"] not in used]
        chosen = rng.sample(pool, min(n, len(pool)))
        used.update(c["clip_id"] for c in chosen)
        picked.extend({**c, "track": track} for c in chosen)
        return len(chosen)

    for split, (n_easy, n_hard) in plan.items():
        pool = [c for c in ok if c["split"] == split]
        take([c for c in pool if not SPECIAL & set(c["tags"])], n_easy, "easy")
        n_rare = take([c for c in pool if RARE & set(c["tags"])], n_hard // 2, "hard")
        take([c for c in pool if not RARE & set(c["tags"])], n_hard - n_rare, "hard")
    return picked


def assign_voices(clip_id):
    """클립 ID로 캐스터 1명 + 해설 1명을 고정 배정 (캐스터 3 × 해설 2 = 6가지 조합)"""
    h = zlib.crc32(clip_id.encode())
    return {"caster": CASTERS[h % len(CASTERS)], "analyst": ANALYSTS[(h // len(CASTERS)) % len(ANALYSTS)]}


picked = sample(clips, PILOT)
for c in picked:
    c["knobs"] = {**KNOBS, **TRACK_KNOBS[c["track"]]}
    c["voices"] = assign_voices(c["clip_id"])
    c["utts"] = render_clip(c, c["knobs"])
    # 의역할 발화 → 기대 라벨 (의역하지 않는 트랙은 비워 둠 → 템플릿 그대로 저장)
    c["todo"] = P.targets(c["utts"], c["events"]) if c["knobs"]["paraphrase"] else {}
    c["repeat_ok"] = P.repeat_ok(c["utts"], c["events"])     # '또 파울'처럼 반복 표현을 써도 사실인 발화

print("캐스터:", CASTERS, "/ 해설:", ANALYSTS)
print(f"{len(picked)}클립, 발화 {sum(len(c['utts']) for c in picked)}개 중 의역 대상 {sum(len(c['todo']) for c in picked)}개 (Hard만)")
print(Counter(l for c in picked for l in c["todo"].values()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
캐스터: ['caster_m35_anchor', 'caster_m38_steady', 'caster_m40_baritone'] / 해설: ['analyst_a40_warm', 'analyst_a45_clear']
20클립, 발화 679개 중 의역 대상 201개 (Hard만)
Counter({'none': 72, 'ball': 38, 'strike': 23, 'foul': 20, 'out': 19, 'reach': 16, 'strikeout': 11, 'walk': 2})


## 1단계: 생성 (Qwen3-32B)

In [ ]:
from vllm import LLM, SamplingParams

ROUNDS = 3          # 첫 생성 + 탈락한 줄 다시 생성 2번
TEMPERATURE = 0.7   # 0.9에서는 '드러눕니다' 같은 이상한 단어 선택이 잦았음

# 32B bf16은 가중치만 약 65GB → 남는 메모리로 KV 캐시. 파일럿·본 데이터 모두 프롬프트가 짧아 충분
# 이미 모델을 띄웠다면(같은 커널에서 다시 실행) 새로 만들지 않는다
if "llm" not in globals():
    with real_stdout():
        llm = LLM(GEN_MODEL, dtype="bfloat16", max_model_len=8192, gpu_memory_utilization=0.93, seed=0)
print("모델 준비 완료:", GEN_MODEL)


def form_check(c, i, new):
    u = c["utts"][i]
    return P.check_form(u["text"], new, u["kind"], repeat_ok=i in c["repeat_ok"], label=c["todo"][i])


accepted = {c["clip_id"]: {} for c in picked}   # clip_id → {발화 번호: 의역}
rejects = Counter()
examples = defaultdict(list)                     # 탈락 이유 → 예시 몇 개
for r in range(ROUNDS):
    jobs = [(c, [i for i in c["todo"] if i not in accepted[c["clip_id"]]]) for c in picked]
    jobs = [(c, todo) for c, todo in jobs if todo]
    if not jobs:
        break
    # 앞 라운드에서 통과한 의역은 문맥으로 보여줘서 말투가 이어지게 한다
    msgs = [P.gen_messages(P.apply(c["utts"], accepted[c["clip_id"]]), set(todo), c["voices"]) for c, todo in jobs]
    params = [SamplingParams(temperature=TEMPERATURE, top_p=0.95, max_tokens=3000,
                             seed=zlib.crc32(f"{c['clip_id']}/{r}".encode())) for c, _ in jobs]
    outs = llm.chat(msgs, params, chat_template_kwargs={"enable_thinking": False})

    n_try = n_ok = 0
    for (c, todo), out in zip(jobs, outs):
        got = P.parse_gen(out.outputs[0].text)
        for i in todo:
            n_try += 1
            new = got.get(i)
            reason = "응답에 없음" if new is None else form_check(c, i, new)
            if reason is None:
                accepted[c["clip_id"]][i] = new
                n_ok += 1
            else:
                rejects[reason] += 1
                if len(examples[reason]) < 3:
                    examples[reason].append(f"{c['utts'][i]['text']}  →  {new}")
    print(f"라운드 {r + 1}: {n_try}줄 중 {n_ok}줄 통과")

total = sum(len(c["todo"]) for c in picked)
done = sum(len(a) for a in accepted.values())
print(f"\n형식 검사 통과 {done}/{total} ({done / total:.0%})")
for reason, n in rejects.most_common():
    print(f"  {n:>4}  {reason}")
    for ex in examples[reason]:
        print(f"         {ex}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Processed prompts: 100%|██████████| 10/10 [00:26<00:00,  2.70s/it, est. speed input: 790.41 toks/s, output: 50.87 toks/s]

라운드 2: 44줄 중 25줄 통과


Rendering conversations:   0%|          | 0/7 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 7/7 [00:11<00:00,  1.58s/it, est. speed input: 1425.63 toks/s, output: 55.68 toks/s]

라운드 3: 19줄 중 12줄 통과

형식 검사 통과 194/201 (97%)
    16  너무 김
         타구가 정면으로 갔네요.  →  타구가 딱 맞아 터지면서 정면으로 날아갔어요. 투수의 컨트롤이 좋았습니다.
         평범한 타구, 아웃.  →  스윙하고 나온 타구가 정면으로 떨어졌습니다. 2루수 손에 잡히고요.
         타구가 정면으로 갔네요.  →  타구가 정면으로 날아가면서 아웃되다 보니, 또 한 번 기회를 놓쳤습니다.
    10  해설에 판정 단어
         유인구에 잘 참았습니다.  →  이번 파울은 잘 참은 투수의 유인구 덕분입니다.
         중요한 순간입니다.  →  이제 이 카운트에서 뭘 할 건가요?
         투수 입장에서는 아쉬운 장면이죠.  →  정면에서 볼을 얻어낸 건데, 투수 입장에서는 아쉬운 볼 배합이에요.
     9  응답에 없음
         볼이고요.  →  None
         땅볼이고요, 잡아서 처리합니다.  →  None
         원아웃 이후의 실점이 제일 뼈아프거든요.  →  None
     8  볼 판정이 아닌데 '볼'
         낮은 공에 헛스윙.  →  낮은 볼에 헛스윙.
         131킬로 포크, 우익선상 3루타!  →  131킬로 포크볼이 우익수 방향으로 날아갑니다. 3루타! 상대 3점째!
         쳤습니다! 안타!  →  피치클락 위반입니다. 볼 하나가 주어집니다.
     8  근거 없는 반복 표현
         헛스윙.  →  다시 헛스윙.
         144킬로 직구, 번트, 희생번트 성공합니다.  →  144킬로 직구, 또 번트 시도입니다. 희생번트로 처리.
         자, 직구, 볼.  →  다시 직구, 이번엔 몸쪽으로 흘러나갑니다. 볼.
     7  없던 숫자
         평범한 타구, 아웃.  →  평범한 타구, 1루수 손에 잡힙니다.
         자, 땅볼이고요, 잡아서 처리합니다.  →  땅볼

In [ ]:
# 1단계 결과 저장 → 커널 재시작 후 2단계에서 읽음
with (OUT / "generated.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        f.write(json.dumps({"clip_id": c["clip_id"], "accepted": accepted[c["clip_id"]]}, ensure_ascii=False) + "\n")

# 클립 하나 미리보기: 원문 → 의역
c = next(c for c in picked if c["track"] == "hard")
acc = accepted[c["clip_id"]]
for i, u in enumerate(c["utts"]):
    who = "해설" if u["speaker"] == "analyst" else "캐스터"
    if i in acc:
        print(f"{who:<3} {u['text']}\n      → {acc[i]}")
    else:
        print(f"{who:<3} {u['text']}" + ("" if i not in c["todo"] else "   (실패 → 원문)"))

캐스터 7회말, 한화 공격입니다. 스코어 SSG 2 대 한화 4.
캐스터 마운드에는 한두솔.
캐스터 노아웃, 타석에는 이도윤.
캐스터 자, 127킬로 슬라이더, 낮은 공 골라냅니다.
      → 127킬로 슬라이더, 낮게 빠집니다.
해설  예, 맞습니다.
캐스터 낮은 볼입니다.
      → 바깥쪽 낮은 볼이네요.
캐스터 직구, 좋은 코스로 빠져나가는 안타입니다.
      → 좋은 코스의 직구, 안타입니다.
해설  가볍게 밀어 쳤네요.
      → 타이밍 좋게 밀어 친 것 같아요.
캐스터 노아웃, 타석에는 최인호.
캐스터 원볼입니다.
해설  초구 스트라이크가 정말 중요하거든요.
      → 첫 공부터 스트라이크 잡는 게 중요하죠.
캐스터 직구, 몸쪽 스트라이크.
      → 몸쪽 직구, 힘 있게 들어옵니다.
해설  이런 상황에서 볼넷을 주면 안 되죠.
      → 이런 상황에서 볼넷은 위험하거든요.
캐스터 낮게 빠졌고요.
      → 바깥쪽으로 살짝 빠집니다.
캐스터 카운트 투앤원.
캐스터 카운트 쓰리볼 원 스트라이크.
해설  앞선 볼도 비슷한 코스였죠.
      → 이런 코스의 볼이 나왔었죠.
캐스터 오른쪽 안타.
      → 오른쪽으로 빠져나가는 안타.
해설  풀카운트까지 가면 투수도 부담이 크죠.
      → 풀카운트까지 가면 투수도 긴장이에요.
캐스터 노아웃, 타석에는 이진영.
캐스터 직구, 낮은 공 골라냅니다.
      → 낮은 직구, 골라냅니다.
캐스터 볼카운트 원앤원.
캐스터 3구 직구, 스트라이크입니다.
      → 3구 직구, 힘 있게 들어옵니다.
캐스터 카운트 원앤투.
캐스터 129킬로 슬라이더, 방망이 헛돌면서 삼진.
      → 129킬로 슬라이더, 헛스윙 삼진!
캐스터 원아웃, 타석에는 페라자.
캐스터 카운트 노볼 원 스트라이크.
해설  아까 그 공은 볼 판정이 조금 아쉬웠어요.
      → 아까 그 공은 볼로 보이지 않았나요?
캐스터 원앤투입니다.
캐스터 볼이고요.
      → 바깥쪽 볼이에요.
캐스터 투앤투

## 2단계: 검증 (Gemma 3 27B)
**커널 재시작 → 셀 2(공통 준비) → 아래 셀** 순서로 실행. (1단계 모델이 GPU 메모리를 잡고 있음)

In [2]:
from getpass import getpass

from vllm import LLM, SamplingParams

# Gemma는 라이선스 동의가 필요한 모델 → HF 토큰 필요
# Colab 웹에서는 보안 비밀(HF_TOKEN)을 읽고, VS Code처럼 보안 비밀을 못 읽는 환경에서는 직접 입력받는다
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        os.environ["HF_TOKEN"] = getpass("Hugging Face 토큰 (Read 권한): ").strip()

with real_stdout():
    llm = LLM(JUDGE_MODEL, dtype="bfloat16", max_model_len=2048, gpu_memory_utilization=0.9, seed=0)
print("모델 준비 완료:", JUDGE_MODEL)

with (OUT / "generated.jsonl").open(encoding="utf-8") as f:
    accepted = {}
    for line in f:
        row = json.loads(line)
        accepted[row["clip_id"]] = {int(i): t for i, t in row["accepted"].items()}

# 1단계 이후에 형식 검사 규칙이 늘었을 수 있으니 지금 규칙으로 한 번 더 거른다 (떨어진 줄은 템플릿 원문)
recheck = Counter()
for c in picked:
    acc = accepted[c["clip_id"]]
    for i in list(acc):
        u = c["utts"][i]
        reason = P.check_form(u["text"], acc[i], u["kind"], repeat_ok=i in c["repeat_ok"], label=c["todo"][i])
        if reason:
            recheck[reason] += 1
            print(f"  재검사 탈락 [{reason}] {u['text']}  →  {acc.pop(i)}")
print("재검사 탈락:", dict(recheck) or "없음")

# 의역문과 함께 원문도 분류한다 → 원문을 못 맞히는 라벨은 검증 모델 자체를 믿기 어렵다는 뜻
items = []  # (클립, 발화 번호, 원문/의역, 기대 라벨)
for c in picked:
    for i, label in c["todo"].items():
        items.append((c, i, "orig", label))
        if i in accepted[c["clip_id"]]:
            items.append((c, i, "para", label))


def utt_of(c, i, which):
    u = c["utts"][i]
    return u if which == "orig" else {**u, "text": accepted[c["clip_id"]][i]}


msgs = [P.judge_messages(utt_of(c, i, which)) for c, i, which, _ in items]
outs = llm.chat(msgs, SamplingParams(temperature=0, max_tokens=5))
preds = [P.parse_judge(o.outputs[0].text) for o in outs]

Hugging Face 토큰 (Read 권한): ··········


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

모델 준비 완료: google/gemma-3-27b-it
  재검사 탈락 [한글 외 문자] 그대로 들어오면서 스트라이크.  →  ตรง구로 들어옵니다, 스트라이크.
재검사 탈락: {'한글 외 문자': 1}


Rendering conversations:   0%|          | 0/394 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 394/394 [00:03<00:00, 118.11it/s, est. speed input: 27674.99 toks/s, output: 350.81 toks/s]


In [3]:
# 검증 결과: 라벨별로 (원문 정확도 = 검증 모델 신뢰도) / (의역 통과율)
# 통과 기준은 P.judge_ok: 판정 발화는 원래 판정과 같아야 하고, 해설은 '정보 없음'이거나 원문과 같은 분류면 통과
orig_pred = {(c["clip_id"], i): pred for (c, i, which, _), pred in zip(items, preds) if which == "orig"}

stat = defaultdict(Counter)
final = {c["clip_id"]: {} for c in picked}
judge_fail = defaultdict(list)
for (c, i, which, label), pred in zip(items, preds):
    stat[label][f"{which}_n"] += 1
    if which == "orig":
        stat[label]["orig_ok"] += pred == label
        continue
    ok = P.judge_ok(label, pred, orig_pred[(c["clip_id"], i)])
    stat[label]["para_ok"] += ok
    if ok:
        final[c["clip_id"]][i] = accepted[c["clip_id"]][i]
    elif len(judge_fail[label]) < 4:
        judge_fail[label].append(f"{c['utts'][i]['text']}  →  {accepted[c['clip_id']][i]}   (분류: {pred})")

print(f"{'라벨':<12} {'원문 정확도':>10} {'의역 통과':>10}")
for label, s in sorted(stat.items()):
    print(f"{label:<12} {s['orig_ok']:>4}/{s['orig_n']:<5} {s['para_ok']:>4}/{s['para_n']:<5}")

total = sum(len(c["todo"]) for c in picked)
done = sum(len(v) for v in final.values())
print(f"\n최종 의역 반영 {done}/{total} ({done / total:.0%}), 나머지는 템플릿 원문")
print("\n검증에서 떨어진 예:")
for label, exs in judge_fail.items():
    print(f"[{label}]")
    for ex in exs:
        print("   ", ex)

라벨               원문 정확도      의역 통과
ball           38/38      33/38   
foul           20/20      16/19   
none           46/72      66/69   
out            19/19      17/17   
reach          16/16      15/15   
strike         23/23      17/22   
strikeout      11/11      11/11   
walk            2/2        2/2    

최종 의역 반영 177/201 (88%), 나머지는 템플릿 원문

검증에서 떨어진 예:
[strike]
    직구, 몸쪽 스트라이크.  →  몸쪽 직구, 힘 있게 들어옵니다.   (분류: ball)
    3구 직구, 스트라이크입니다.  →  3구 직구, 힘 있게 들어옵니다.   (분류: ball)
    직구, 높은 스트라이크.  →  직구, 높게 빠지네요.   (분류: ball)
    초구, 스트라이크.  →  첫 공, 빠르게 들어갑니다.   (분류: ball)
[ball]
    일단 빠집니다.  →  비슷한 코스로 들어옵니다.   (분류: none)
    일단 빠집니다.  →  타구는 중간으로 빠집니다.   (분류: out)
    초구 직구, 일단 빠집니다.  →  빠르게 들어오는 직구, 흔들리지 않고 빠집니다.   (분류: strike)
    일단 빠집니다.  →  타구가 왼쪽으로 빠집니다.   (분류: reach)
[none]
    투수가 원하는 대로 유도했습니다.  →  투수가 정확하게 유도했죠.   (분류: strike)
    투수가 잘 막아냈어요.  →  타격 포인트를 완전히 빗나갔죠. 잘 막아낸 투구입니다.   (분류: strike)
    초구 스트라이크가 정말 중요하거든요.  →  초구 스트라이크는 기분 좋은 시작점이죠.   (분류: strike)
[foul]
    직구

In [4]:
# 저장: 최종 대본(원문은 orig에 보존) + 보고서
import importlib.metadata as md

with (OUT / "scripts.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        row = {k: c[k] for k in ("clip_id", "split", "track", "tags", "voices", "knobs")}
        row["utts"] = P.apply(c["utts"], final[c["clip_id"]])
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

report = {
    "gen_model": GEN_MODEL, "judge_model": JUDGE_MODEL, "vllm": md.version("vllm"),
    "targets": total, "applied": done,
    "by_label": {label: dict(s) for label, s in stat.items()},
}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=1))

{
 "gen_model": "Qwen/Qwen3-32B",
 "judge_model": "google/gemma-3-27b-it",
 "vllm": "0.31.0",
 "targets": 201,
 "applied": 177,
 "by_label": {
  "ball": {
   "orig_n": 38,
   "orig_ok": 38,
   "para_n": 38,
   "para_ok": 33
  },
  "reach": {
   "orig_n": 16,
   "orig_ok": 16,
   "para_n": 15,
   "para_ok": 15
  },
  "none": {
   "orig_n": 72,
   "orig_ok": 46,
   "para_n": 69,
   "para_ok": 66
  },
  "strike": {
   "orig_n": 23,
   "orig_ok": 23,
   "para_n": 22,
   "para_ok": 17
  },
  "strikeout": {
   "orig_n": 11,
   "orig_ok": 11,
   "para_n": 11,
   "para_ok": 11
  },
  "foul": {
   "orig_n": 20,
   "orig_ok": 20,
   "para_n": 19,
   "para_ok": 16
  },
  "out": {
   "orig_n": 19,
   "orig_ok": 19,
   "para_n": 17,
   "para_ok": 17
  },
  "walk": {
   "orig_n": 2,
   "orig_ok": 2,
   "para_n": 2,
   "para_ok": 2
  }
 }
}


In [5]:
# 사람 점검용: 반영된 의역을 라벨별로 무작위 몇 개씩
rng = random.Random(0)
by_label = defaultdict(list)
for c in picked:
    for i, t in final[c["clip_id"]].items():
        by_label[c["todo"][i]].append(f"{c['utts'][i]['text']}  →  {t}")
for label, rows in sorted(by_label.items()):
    print(f"[{label}] {len(rows)}개")
    for row in rng.sample(rows, min(6, len(rows))):
        print("   ", row)

[ball] 33개
    체인지업, 몸쪽 볼입니다.  →  체인지업, 몸쪽으로 날아오지만 볼입니다.
    바깥쪽 공 골라냅니다.  →  바깥쪽 공, 조용히 골라냅니다.
    낮은 볼입니다.  →  바깥쪽 낮은 볼이네요.
    낮은 공 골라냅니다.  →  낮은 공 골라냅니다.
    낮은 볼입니다.  →  바깥쪽 낮은 공이 들어옵니다. 볼입니다.
    낮은 공 골라냅니다.  →  아랫쪽 공, 골라내고 서 있습니다.
[foul] 16개
    커트해냅니다.  →  타구가 빗나가면서 파울이 됩니다.
    자, 커브, 파울이 됐고요.  →  커브로 끝내려 했는데 파울입니다.
    파울.  →  파울.
    빗맞으면서 파울.  →  타구는 빗나가면서 뒤쪽으로 날아갑니다. 파울입니다.
    151킬로 직구, 파울.  →  151킬로 직구, 다시 바깥쪽 파울.
    파울이 됐고요.  →  다시 파울, 조금 더 왼쪽으로 빗나갑니다.
[none] 66개
    이런 상황에서 볼넷을 주면 안 되죠.  →  이런 코스에서는 볼넷을 주면 집중력이 흔들리죠.
    아까 그 공은 볼 판정이 조금 아쉬웠어요.  →  이 공은 볼로 봤지만 살짝 아쉬운 판정이었어요.
    투 스트라이크 이후에는 아무래도 타자가 불리해지죠.  →  스트라이크가 나가면 타자 입장에서는 압박이 커지죠. 공을 고르기 힘들어요.
    타구가 정면으로 갔네요.  →  타구의 방향이 투수 쪽으로 향한 게 눈에 띄었어요.
    삼진 욕심보다는 맞혀 잡는 게 좋습니다.  →  볼넷은 이닝을 이어갈 수 있는 좋은 카드인데, 삼진을 너무 노리는 건 위험하죠.
    삼진 욕심보다는 맞혀 잡는 게 좋습니다.  →  맞혀서 아웃시키는 게 더 안정적이죠.
[out] 17개
    땅볼이고요, 잡아서 처리합니다.  →  정면으로 떨어졌고요, 손에 잡힙니다.
    자, 3구 146킬로 직구, 땅볼이고요, 잡아서 처리합니다.  →  146킬로 직구, 타격은 땅볼로 끝납니다. 수비수가 여유 있게 잡습니다.
  